# Should a credit guarantee program screen applicants by productivity?

**Policy question.** Credit guarantee schemes for small and medium firms usually face one design choice early on: screen applicants by productivity/potential, or extend credit on a first-come basis. This notebook builds a small general equilibrium (GE) model to inform that choice.

<small>*Note: A general equilibrium model makes it possible to compare program designs before they are implemented or scaled. Because a national credit program affects wages and the cost of capital for the whole economy — not only for the firms it supports — the model solves for these economy-wide prices directly rather than holding them fixed. This matters because a design that looks effective when evaluated only on the firms it supports can deliver a negative result once its effect on economy-wide prices is taken into account.*</small>

**The model.** Each firm has two separate characteristics: its productivity (`s`), and its net worth (`a`) — the collateral it can pledge to a lender. These are independent; a firm is not more talented just because it is wealthier. A lender will only extend a firm enough credit to finance capital up to `k ≤ λ·a` — that is, capital cannot exceed a fixed multiple (`λ`) of the firm's own collateral (`a`), regardless of how productive the firm is. This means a highly productive firm with little net worth can be held below the scale it would otherwise choose — not because it lacks the talent to grow, but because no lender will finance the capital required to get there.

We solve for the economy-wide price and firm-size distribution that this creates, then compare aggregate output under four different values of `λ`: a uniform baseline, and three policies that raise `λ` for a specific group of firms, loosening the cap on how much they can borrow against the collateral they already have.

## Model anatomy

The model has five building blocks. Each maps to at least one equation, and the equations are solved in a specific order — the **cadence**. Reading the model through this lens makes it easier to follow what each step of the code is doing.

---

**Block 1 — Technology.** A firm with productivity `s` combines capital and labor to produce output:
$$y = s\,k^{\theta_k}n^{\theta_n}$$
Decreasing returns ($\theta_k + \theta_n < 1$) mean larger firms face diminishing gains — this keeps the firm-size distribution from collapsing to a single dominant firm. Productivity itself follows a persistent AR(1) process in logs, discretized into a finite grid (Step 1):
$$\log s' = \rho \log s + \varepsilon, \quad \varepsilon \sim N(0,\sigma^2)$$

---

**Block 2 — The constrained firm (Step 3).** Each period, a firm picks capital and labor to maximize profit, but it cannot borrow beyond a fixed multiple of its own collateral:
$$\max_{k,n} \;\; p\,s\,k^{\theta_k}n^{\theta_n} - rk - wn \quad\text{subject to}\quad k \le \lambda a$$
When the constraint binds, a talented firm produces less than it would choose to — not because of poor management, but because no lender will finance the capital it needs. Relaxing `λ` for those firms is exactly what the credit guarantee does.

---

**Block 3 — The dynamic exit decision (Step 4).** A firm weighs its profit today against the option of continuing tomorrow. It stays only if the value of continuing exceeds the value of exiting (zero):
$$V(s,a) = \pi(s,a\,;p) - c_f + \beta\,\mathbb{E}\big[\max\{0,\,V(s',a)\}\mid s\big]$$
The `max{0, V}` term is the exit option: a firm that expects to lose money shuts down rather than accumulate losses. The fixed operating cost $c_f$ means even a productive firm will exit if its capital is constrained so tightly that profits cannot cover it.

---

**Block 4 — Aggregation (Step 5).** Individual exit and entry decisions, summed across all firms, determine how the population of firms evolves. A stationary equilibrium requires this distribution to reproduce itself exactly, period after period:
$$\mu'(s',a) = \sum_{s} \mathbb{1}\{V(s,a)\ge 0\}\, Q(s'\mid s)\,\mu(s,a) \;+\; M \cdot \text{entryweight}(s',a)$$
The left side is next period's distribution; the right side is today's survivors (those with $V \ge 0$, evolving through the transition matrix $Q$) plus new entrants scaled by the entry mass $M$.

---

**Block 5 — Market clearing (Step 5).** Two conditions pin down the two unknowns. Free entry pins down the output price `p`: entering is exactly break-even, no windfall profits and no losses:
$$\mathbb{E}_{(s_0,a)}\big[V(s_0,a)\big] = c_e$$
Labor-market clearing pins down the mass of active firms `M`: total labor demanded equals the fixed labor supply:
$$\int n(s,a)\, d\mu(s,a) = L$$

---

**The cadence — the order this is actually solved in.** The free-entry condition is the only one of the five blocks that does not depend on `M`, so it is solved first: `p` is adjusted until free entry holds exactly. Once `p` is pinned down, the stationary distribution `μ` is computed at that price. Only then is the labor-market condition used — not to re-solve anything, but simply to read off the `M` consistent with everything already found.

> Price first → distribution second → entrant mass last.

Each step uses only what the previous one already determined. The four policy regimes in Step 6 each solve this same system, differing only in `λ` and which firms receive it.

---

*Two model simplifications worth flagging upfront. Capital is supplied from outside at a fixed rental rate `r` — there is no capital-market-clearing condition, so the interest rate does not respond to credit policy. And the wage `w = 1` is the numeraire; every other price is expressed relative to it.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.stats import norm

np.set_printoptions(precision=3, suppress=True)

## Step 1 — discretize productivity

Productivity follows a persistent AR(1) in logs: `log(s') = ρ·log(s) + ε`. A computer can't solve over a continuum, so Tauchen's method (1986) approximates it with a finite grid plus a transition matrix `Q` — turning an integral into linear algebra.

In [ ]:
def tauchen(n, rho, sigma, m=3.0):
    sd_z = sigma / np.sqrt(1 - rho**2)
    z_grid = np.linspace(-m * sd_z, m * sd_z, n)
    step = z_grid[1] - z_grid[0]
    Q = np.zeros((n, n))
    for i in range(n):
        mu = rho * z_grid[i]
        for j in range(n):
            if j == 0:
                Q[i, j] = norm.cdf((z_grid[0] - mu + step / 2) / sigma)
            elif j == n - 1:
                Q[i, j] = 1 - norm.cdf((z_grid[-1] - mu - step / 2) / sigma)
            else:
                Q[i, j] = (norm.cdf((z_grid[j] - mu + step/2) / sigma)
                           - norm.cdf((z_grid[j] - mu - step/2) / sigma))
    return z_grid, Q / Q.sum(axis=1, keepdims=True)

## Step 2 — calibration (stylized)

Parameters are set to plausible, literature-typical values (Hopenhayn & Rogerson, 1993-style persistence and volatility), **not estimated from a specific country's data**. See *Limitations* at the end for what an operational version would need.

In [ ]:
N_Z, RHO, SIGMA = 25, 0.90, 0.20          # productivity process
THETA_K, THETA_N = 0.30, 0.35              # capital & labor curvature
R_RATE, BETA = 0.10, 0.85                  # capital rental rate, discount factor
CF, CE, L_SUPPLY = 0.55, 1.10, 1.0         # fixed operating cost, entry cost, labor supply

LAMBDA_BASE, LAMBDA_BOOST = 1.5, 5.0       # collateral multiplier: normal vs. policy-favored
A_VALS = np.array([0.4, 0.8, 1.5, 3.0, 6.0])  # net worth types (independent of talent)

z_grid, Q = tauchen(N_Z, RHO, SIGMA)
s_grid = np.exp(z_grid)

# entrants are drawn from the lower part of the productivity grid
entry_logits = -np.maximum(z_grid, 0) * 2.0 - 0.5 * z_grid**2
nu = np.exp(entry_logits); nu = nu / nu.sum()

## Step 3 — the firm's problem

Each period a firm picks capital and labor to maximize profit, subject to `k ≤ λ·a` (Block 2 above). The unconstrained optimum has a closed form; if it exceeds the collateral ceiling, capital is capped there and labor is re-optimized around it.

In [ ]:
def static_choice_credit(p, s, k_max, theta_k=THETA_K, theta_n=THETA_N, r=R_RATE, w=1.0):
    nu_rts = theta_k + theta_n
    gamma = (r * theta_n) / (w * theta_k)
    k_u = (np.maximum(p * s * theta_k * gamma**theta_n, 1e-12) / r) ** (1 / (1 - nu_rts))
    n_u = k_u * gamma

    constrained = k_u > k_max
    k = np.where(constrained, k_max, k_u)
    n_c = (np.maximum(p * s * theta_n * np.maximum(k, 1e-12)**theta_k / w, 1e-12)) ** (1 / (1 - theta_n))
    n = np.where(constrained, n_c, n_u)

    output = s * np.maximum(k, 1e-12)**theta_k * np.maximum(n, 1e-12)**theta_n
    profit = p * output - r * k - w * n
    return k, n, profit, output, constrained

## Step 4 — the dynamic exit decision

A firm stays only if continuing is worth more than exiting (Block 3 above). The value function is solved by iteration on the Bellman equation — repeated application converges because the Bellman operator is a contraction.

In [ ]:
def solve_value_credit(p, k_max, cf=CF, beta=BETA, tol=1e-9, maxit=3000):
    k, n, profit, output, constrained = static_choice_credit(p, s_grid, k_max)
    V = np.zeros(N_Z)
    for _ in range(maxit):
        V_new = profit - cf + beta * (Q @ np.maximum(V, 0.0))
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new
    return V, V < 0, k, n, profit, output, constrained

## Step 5 — closing the model

Two conditions pin down the equilibrium (Blocks 4 and 5 above). **Free entry** adjusts the price `p` until entering is exactly break-even. **Labor-market clearing** scales the mass of active firms `M` until total labor demand equals the fixed labor supply. Following the cadence: price is solved first (by root-finding), then the stationary distribution is computed at that price, then `M` is read off from labor clearing.

In [ ]:
def stationary_dist(exit_policy, entry_mass, tol=1e-12, maxit=20000):
    stay = (~exit_policy).astype(float)
    mu = entry_mass.copy()
    for _ in range(maxit):
        mu_new = Q.T @ (stay * mu) + entry_mass
        if np.max(np.abs(mu_new - mu)) < tol:
            return mu_new
        mu = mu_new
    return mu

def solve_equilibrium(k_max_vals, entry_weight, p_bracket=(0.05, 80.0)):
    def free_entry_gap(p):
        return sum(np.sum(w * solve_value_credit(p, k)[0])
                   for k, w in zip(k_max_vals, entry_weight)) - CE
    p_star = brentq(free_entry_gap, *p_bracket, xtol=1e-10)

    labor_d = output = mass = exits = constrained_mass = 0.0
    for k_max, w in zip(k_max_vals, entry_weight):
        V, exit_pol, k, n, profit, out, constr = solve_value_credit(p_star, k_max)
        mu = stationary_dist(exit_pol, w)
        labor_d += np.sum(mu * n); output += np.sum(mu * out)
        mass += np.sum(mu); exits += np.sum(mu * exit_pol)
        constrained_mass += np.sum(mu * constr)

    M = L_SUPPLY / labor_d
    return dict(p=p_star, aggregate_output=M * output, mass=M * mass,
                exit_rate=exits / mass, constrained_share=constrained_mass / mass)

## Step 6 — four policy regimes

1. **Frictionless** — no borrowing limit (upper-bound benchmark).
2. **Constrained, no policy** — baseline collateral multiplier `λ`, wealth drawn independently of talent.
3. **Targeted** — the guarantee raises `λ` for the top-quintile productivity entrants.
4. **Mistargeted** — the same guarantee, but aimed at the bottom quintile instead.

In [ ]:
def run(regime):
    if regime == "frictionless":
        return solve_equilibrium([1e6], [nu.copy()])

    n_a = len(A_VALS)
    if regime == "constrained":
        return solve_equilibrium(list(LAMBDA_BASE * A_VALS),
                                  [(1/n_a) * nu.copy() for _ in range(n_a)])

    cdf = np.cumsum(nu)
    favored = (cdf > 0.80) if regime == "targeted" else (cdf <= 0.20)
    k_max_vals, entry_weight = [], []
    for a in A_VALS:
        k_max_vals.append(LAMBDA_BASE * a)
        entry_weight.append(np.where(~favored, nu, 0.0) / n_a)
        k_max_vals.append(LAMBDA_BOOST * a)
        entry_weight.append(np.where(favored, nu, 0.0) / n_a)
    return solve_equilibrium(k_max_vals, entry_weight)

REGIMES = ["frictionless", "constrained", "targeted", "mistargeted"]
results = {r: run(r) for r in REGIMES}
print("solved all four regimes")

## Results

In [ ]:
Y_f = results["frictionless"]["aggregate_output"]
table = pd.DataFrame({
    r: {"output (% of frictionless)": 100 * results[r]["aggregate_output"] / Y_f,
        "exit rate": results[r]["exit_rate"],
        "share constrained": results[r]["constrained_share"]}
    for r in REGIMES
}).T
print(table.round(3))

Targeting the guarantee at high-productivity entrants closes roughly half the output gap to the frictionless benchmark; mistargeting it leaves output unchanged from doing nothing at all.

In [ ]:
labels = ["Frictionless", "Constrained,\nno policy", "Targeted\ncredit access", "Mistargeted\ncredit access"]
vals = [100 * results[r]["aggregate_output"] / Y_f for r in REGIMES]
colors = ["#2E75B6", "#8C8C8C", "#2E9E5B", "#C0392B"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, vals, color=colors)
ax.axhline(100, color="grey", lw=0.8, ls="--")
ax.set_ylabel("Aggregate output\n(frictionless = 100)")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 2, f"{v:.1f}", ha="center")
ax.set_title("Targeting recovers about half the output lost to the financing friction")
plt.tight_layout()
plt.show()

The mechanism behind that result: targeting relaxes the constraint specifically where it binds. Mistargeting relaxes it where it never bound in the first place — it doesn't reduce output, it just doesn't do anything.

In [ ]:
vals2 = [100 * results[r]["constrained_share"] for r in REGIMES]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, vals2, color=colors)
ax.set_ylabel("Share of active firms\nthat are capital-constrained (%)")
for b, v in zip(bars, vals2):
    ax.text(b.get_x() + b.get_width()/2, v + 1.5, f"{v:.1f}", ha="center")
ax.set_title("Targeting relaxes the constraint where it binds; mistargeting doesn't touch it")
plt.tight_layout()
plt.show()

## Policy takeaway

- Targeting the guarantee at productive-but-constrained firms recovers about half the output lost to the financing friction.
- Mistargeting it is *wasted*, not *harmful* — those firms have no productive use for the extra borrowing room.
- Practical implication: **how the guarantee is screened matters more than how large it is.** A well-targeted, modestly funded program can outperform a generously funded, untargeted one.
- The hard part the model assumes away: actually observing which firms are talented-but-poor versus simply unproductive.

## Limitations

- Calibration is stylized — plausible, literature-typical values, not estimated from a real country's firm data.
- Net worth is a fixed type at entry, not accumulated through saving (a full Buera-Kaboski-Shin model would relax this).
- Cross-sectional, steady-state comparison only — no aggregate shocks or transition dynamics.

## Next step toward an operational version

Replace the stylized productivity and net-worth distributions with moments estimated from World Bank Enterprise Survey data for a specific country, and recalibrate `λ` to match observed firm-level borrowing constraints.